# 01 Data Exploration

Exploring OpenFDA and RxNorm data formats.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

project_root = os.chdir("..")
sys.path.insert(0, str(project_root))

os.getcwd()

In [ ]:
import pandas as pd
import json

In [ ]:
from src.knowledge_graph.openfda_connector import OpenFDAConnector
from src.knowledge_graph.rxnorm_connector import RxNormConnector
from src.knowledge_graph.rxnorm_extractor import RxNormExtractor

## Data

In [ ]:
# Load the list of top 300 drugs from ClinCalc
top_300_drugs = pd.read_csv("data/raw/clincalc_top_300_drugs_2023.csv")
top_300_drugs.head()

In [ ]:
# Select a random drug name from the list
random_drug = top_300_drugs.sample(1).iloc[0].drug_name

# Take the first name if there are multiple
if ";" in random_drug:
    random_drug = random_drug.split(";")[0]  

print(f"Selected drug: {random_drug}")

## OpenFDA Exploration

### Test API

In [ ]:
openfda_connector = OpenFDAConnector()

In [ ]:
openfda_result = openfda_connector.test_api("aspirin")

### Query by Generic Name

In [ ]:
results = openfda_connector.query_api_for_generic_name("aspirin", search_limit=5)
len(results)

In [ ]:
results = openfda_connector.query_api_for_generic_name(random_drug)

# Display the first result
if len(results) > 0:
    first_result = results[0]
    print("First result:")
    print(f"Brand name: {first_result['openfda']['brand_name']}")
    print(f"Generic name: {first_result['openfda']['generic_name']}")
    print(f"Manufacturer name: {first_result['openfda']['manufacturer_name']}")
    print(json.dumps(first_result, indent=2)[:1000])  # First 1000 chars

In [ ]:
if first_result:
    print("Available keys:")
    available_keys = first_result.keys()
    for key in available_keys:
        print(key)

In [ ]:
for key in available_keys:
    print(f"\nDetails for '{key}':")
    if isinstance(first_result[key], list):
        for item in first_result[key]:
            print(f"  {item}")
    else:
        print(f"  {first_result[key]}")

In [ ]:
selected_key = 'openfda'

if selected_key in first_result:
    print(f"\nDetails for '{selected_key}':")
    for key in first_result[selected_key]:
        print(f"  {key}: {first_result[selected_key][key]}")

In [ ]:
first_result_rxcui = first_result['openfda'].get('rxcui', [])[0]
print(f"\nRXCUI: {first_result_rxcui}")

### Query by RXCUI

In [ ]:
results = openfda_connector.fetch_by_rxcui(first_result_rxcui)
results

### Extract structured fields

In [ ]:
structured_label = openfda_connector.extract_structured_fields(first_result)
structured_label

In [ ]:
extracted_entities = openfda_connector.extract_entities_from_label(structured_label)
extracted_entities

## RxNorm Exploration

In [ ]:
# Test connection
rxnorm = RxNormConnector()
rxnorm_connected = rxnorm.connect()

if rxnorm_connected:
    rxnorm_result = rxnorm.test_query("aspirin")
    rxnorm.close()

In [ ]:
# Test extractor
rxnorm_connected = RxNormConnector()
rxnorm_connected.connect()

rxnorm_extractor = RxNormExtractor(rxnorm_connected.connection)

drug_name = "aspirin"
#drug_name = "aspirin 400 MG"
# drug_name = "ACETYLSALICYLIC ACID"
drug_name =  random_drug

rxcui = rxnorm_extractor.find_drug_rxcui(drug_name)
print(f"RXCUI for {drug_name}: {rxcui}")

ingredients = rxnorm_extractor.get_ingredients(rxcui)
ingredient_names = [ing['name'] for ing in ingredients]
print(f"Ingredients for {drug_name} (RXCUI: {rxcui}): {ingredient_names}")

related_drugs = rxnorm_extractor.is_ingredient_of(rxcui)
print(f"Drugs that contain {drug_name} (RXCUI: {rxcui}): {[drug['name'] for drug in related_drugs]}")

aliases = rxnorm_extractor.get_aliases(rxcui)
print(f"Aliases for {drug_name} (RXCUI: {rxcui}): {aliases}")

In [ ]:
drug_info = rxnorm_extractor.extract_drug(drug_name)
drug_info

## RXCUI Match Exploration

In [ ]:
drug_list_raw = top_300_drugs.drug_name.to_list()
print(f"Initial number of drug entries: {len(drug_list_raw)}")

# Seprate entries with ";" into individual drugs
for drug in drug_list_raw:
    if ";" in drug:
        drugs = drug.split(";")
        drug_list_raw.extend(drugs)
        drug_list_raw.remove(drug)
print(f"Number of drug entries after splitting: {len(drug_list_raw)}")
        
# Remove duplicates
drug_list_raw = list(set(drug_list_raw))
print(f"Number of unique drug entries after deduplication: {len(drug_list_raw)}")

In [ ]:
openfda_connector = OpenFDAConnector()

In [ ]:
top_300_drugs_rxcui = []

for drug in drug_list_raw:
    print(f"\nProcessing drug: {drug}")
    results = openfda_connector.query_api_for_generic_name(drug, search_limit=1000)
    
    rxcui_list = []

    # Extract RXCUIs from results
    for result in results:
        rxcui = result['openfda'].get('rxcui', None)
        
        if rxcui:
            if len(rxcui) > 1:
                rxcui_list.extend(rxcui)
            else:
                rxcui_list.append(rxcui[0])
    print(f"Number of RXCUIs found: {len(rxcui_list)}")

    # Drop duplicates
    rxcui_list = list(set(rxcui_list))
    print(f"Number of unique RXCUIs: {len(rxcui_list)}")

    # Add the unique RXCUIs to the main list
    top_300_drugs_rxcui.extend(rxcui_list)

print(f"\nTotal RXCUIs for top 300 drugs: {len(top_300_drugs_rxcui)}")

# Drop duplicates from the main list
top_300_drugs_rxcui = list(set(top_300_drugs_rxcui))
print(f"\nTotal unique RXCUIs for top 300 drugs: {len(top_300_drugs_rxcui)}")

In [ ]:
print(f"\nTotal unique RXCUIs for top 300 drugs: {len(top_300_drugs_rxcui)}")

In [ ]:
rxcui_list = []

# Extract RXCUIs from results
for result in results:
    rxcui = result['openfda'].get('rxcui', None)
    
    if rxcui:
        if len(rxcui) > 1:
            rxcui_list.extend(rxcui)
        else:
            rxcui_list.append(rxcui[0])
print(f"Number of RXCUIs found: {len(rxcui_list)}")

# Drop duplicates
rxcui_list = list(set(rxcui_list))
print(f"Number of unique RXCUIs: {len(rxcui_list)}")

In [ ]:
rxnorm_connected = RxNormConnector()
rxnorm_connected.connect()

rxnorm_extractor = RxNormExtractor(rxnorm_connected.connection)

In [ ]:
rxconso = rxnorm_extractor.get_rxnconso_df()
rxconso.shape

In [ ]:
rxconso.head()

In [ ]:
import random


In [ ]:
random_rxcui = random.sample(top_300_drugs_rxcui, 1)
rxconso[rxconso['RXCUI'] == random_rxcui[0]]